## 01 - Hybrid - Pitt
In this notebook, we will try a hybrid (joint) architecture with Pitt set.
This model contains:
- Tfidf (as in classical model)
- Bert embeddings (using sentence-transformer library)
- Bert summzarizer model
- Logistic regression classifier


In [149]:
!pip install sentence-transformers==2.7.0
!pip install bert-extractive-summarizer==0.10.1

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [199]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report, mean_squared_error
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import cross_val_score
from sentence_transformers import SentenceTransformer
from summarizer import Summarizer
from sklearn.pipeline import make_pipeline

In [151]:
# some main options about using Tfidf, sentence embeddings and summary
train_ratio = 1.0             # the fraction of training set to use. Set to 1.0 for using all train data
use_summary = True            # set to False to skip summarization text
embed_model_name = "LaBSe"    # LaBSe model provides good performance and efficiency, supports multi-language
concat_vector = True          # combine Tfidf + embeddings. Set to False to use Tfidf only, ignoring Bert sentence embeddings
use_embeddings_only = False   # set to True to use Bert embedding only, ignoring Tfidf
tuning_score = "f1-score"     # can be recall / precision / f1-score. Use for hyper-parameter tuning

In [152]:
# load Pitt set

# please point to correct downloaded transcribed folders of the original dataset
data_path = "../../../data/" 

train_fname = data_path + "pitt/pitt_train_txt.csv"
dev_fname = data_path + "pitt/pitt_dev_txt.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 337
Dev df = 115


In [153]:
# in low-resource setting, maybe it's good to use a smaller subset of train data 
# to prevent overfitting on train set

train_df = train_df.sample(frac=train_ratio, random_state=42)
train_df.reset_index(inplace=True)
print(len(train_df))

337


In [154]:
train_df.head()

,index,file_id,target,text,mmse,id,split_name
0,46,120-1,AD,okay .\nokay .\nokay .\nwell ‡ the boy is <rea...,28.0,120,train
1,163,579-0,AD,woman dish\nboy climbin(g) up to get some cook...,18.0,579,train
2,57,168-0,AD,uh the &+b boy's &+f gonna fall off of the &-...,20.0,168,train
3,25,065-0,AD,um a &+g a girl is putting her finger up to h...,29.0,65,train
4,113,334-0,AD,stool <is being>\nthe little girl is &-uh crit...,20.0,334,train


In [155]:
dev_df.head()

,file_id,target,text,mmse,id,split_name
0,016-0,AD,guy or a boy reaching for a cookie jar &-uh ti...,28.0,16,dev
1,016-1,AD,uh the water's spilling over .\ndrying the dis...,NaN,16,dev
2,016-3,AD,a boy is reaching for the cookie jar and tripp...,NaN,16,dev
3,016-4,AD,well the little girl wants the cookie that the...,NaN,16,dev
4,033-0,AD,okay .\napparently a mother and a daughter and...,26.0,33,dev


### Pre-processing
We will lowercase the text for better efficiency

In [156]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

In [157]:
train_df.head()

,index,file_id,target,text,mmse,id,split_name,text_lower
0,46,120-1,AD,okay .\nokay .\nokay .\nwell ‡ the boy is <rea...,28.0,120,train,okay .\nokay .\nokay .\nwell ‡ the boy is <rea...
1,163,579-0,AD,woman dish\nboy climbin(g) up to get some cook...,18.0,579,train,woman dish\nboy climbin(g) up to get some cook...
2,57,168-0,AD,uh the &+b boy's &+f gonna fall off of the &-...,20.0,168,train,uh the &+b boy's &+f gonna fall off of the &-...
3,25,065-0,AD,um a &+g a girl is putting her finger up to h...,29.0,65,train,um a &+g a girl is putting her finger up to h...
4,113,334-0,AD,stool <is being>\nthe little girl is &-uh crit...,20.0,334,train,stool <is being>\nthe little girl is &-uh crit...


In [158]:
dev_df.head()

,file_id,target,text,mmse,id,split_name,text_lower
0,016-0,AD,guy or a boy reaching for a cookie jar &-uh ti...,28.0,16,dev,guy or a boy reaching for a cookie jar &-uh ti...
1,016-1,AD,uh the water's spilling over .\ndrying the dis...,NaN,16,dev,uh the water's spilling over .\ndrying the dis...
2,016-3,AD,a boy is reaching for the cookie jar and tripp...,NaN,16,dev,a boy is reaching for the cookie jar and tripp...
3,016-4,AD,well the little girl wants the cookie that the...,NaN,16,dev,well the little girl wants the cookie that the...
4,033-0,AD,okay .\napparently a mother and a daughter and...,26.0,33,dev,okay .\napparently a mother and a daughter and...


### Apply summarizer model to shorten the text

In [159]:
# initialize summarization model
sum_model = Summarizer()

In [160]:
import warnings
# hide warnings
warnings.filterwarnings('ignore')

In [161]:
# test text summarization
s = """
"now wait a minute .
everything that's going on in the picture just by looking at it ?
well, she's wiping dishes .
and the boy is gettin(g) cookies out_of a jar .
and he's on a stool .
and there's cups over here .
and the water's going on the floor .
and the housewife, the mother, is wiping the dishes .
and she's standing by the window .
and &-uh she's lettin(g) the water run out_of the faucet onto the
and the  &+k the ladder is fallin(g) over when he's gettin(g) the
what else you want ?",AD
355-1,"well ‡ the boy is falling .
he's trying to get something  a cookie jar .
and <the stool is>  (.) he's gonna be on the floor in a minute
and &-uh he is tryin(g) to get a cookie for &+h this gal .
and she's over here &-uh +...
oh ‡ &+sh she's not payin(g) any attention .
and she didn't turn off the water .
and the water is drippin(g) all over the floor .
do you see that ?
and she's trying to &-uh +...
=laughs she's in a mess .
she's got <in a>  &+f water all over the floor .
and she's standing in the water .
and &-uh she has an apron on .
and she's tryin(g) to wipe the plates dry .
and &-uh (...) +..."
"""
summary = sum_model(s)
print(summary)

everything that's going on in the picture just by looking at it ? and the housewife, the mother, is wiping the dishes . and she's tryin(g) to wipe the plates dry .


In [162]:
if use_summary:
    summ_f = lambda text: f"{text.lower()}. Summary: {sum_model(text).lower()}"
else:
    summ_f = lambda text: f"{text.lower()}" # use it to skip summarization

In [163]:
# apply summary to dataframes
print("Performing summarization")
print("train set...")

train_df["text_lower"] = train_df["text"].apply(summ_f)

print("dev set...")
dev_df["text_lower"] = dev_df["text"].apply(summ_f)

print("Completed")

Performing summarization
train set...
dev set...
Completed


### Build Tfidf Vectorizer

We can use Tfidf Vectorizer to convert text to a fixed-length vector

In [164]:
vectorizer = TfidfVectorizer()

corpus = train_df["text_lower"]
print(len(corpus))
print("Building Tfidf Vectorizer...")
X_train_tfidf = vectorizer.fit_transform(corpus)
print("Completed")

337
Building Tfidf Vectorizer...
Completed


In [165]:
print(X_train_tfidf.shape)

(337, 1438)


### Create transformer-based sentence embeddings


In [166]:
# initialize sentence embeddings model
vectorizer_embed = SentenceTransformer(embed_model_name) 

In [167]:
corpus = train_df["text_lower"]
print("Creating text embeddings...")
X_train_embed = vectorizer_embed.encode(corpus)
print("Completed")

Creating text embeddings...
Completed


In [168]:
print(type(X_train_embed))
print(X_train_embed.shape)

<class 'numpy.ndarray'>
(337, 768)


In [169]:
# double check Tfidf vector
print(type(X_train_tfidf))
print(X_train_tfidf.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(337, 1438)


#### Concate 2 vectors (Tfidf + sentence embeddings)


In [170]:
# test vector concatenation 
v1 = [[2, 4, 6],
      [1, 3, 5]
     ]
v2 = [[8, 10],
      [7, 9]
     ]
v3 = np.concatenate((v1, v2), axis=1) # concate on horizontal axis
print(v3) # should be [2, 4, 6, 8, 10], [1, 3, 5, 7, 9]
assert np.allclose(v3,np.array([[2, 4, 6, 8, 10], [1, 3, 5, 7, 9]]))
print("Passed")

[[ 2  4  6  8 10]
 [ 1  3  5  7  9]]
Passed


In [171]:
def combine_vectors(tfidf_vec, embed_vec):
    """
        combine vectors using global settings (concat_vecor and use_embeddings_only)
    """
    if concat_vector:
        result = np.concatenate((tfidf_vec.toarray(), embed_vec), axis=1)
    elif use_embeddings_only:
        result = embed_vec
    else: # use tfidf only
        result = tfidf_vec

    return result

In [172]:
X_train = combine_vectors(X_train_tfidf, X_train_embed)
print(X_train.shape)

(337, 2206)


### Train Logistic regression using combined vector

We can now train a simple logistic regression classifier using processed X_train

In [173]:
y_train = train_df["target"]

print("Training logistic classifier...")

clf = LogisticRegression(random_state=42,
                         class_weight= "balanced" 
                        ).fit(X_train, y_train)

print("Completed")

Training logistic classifier...
Completed


In [174]:
# getting score on train set
train_score = clf.score(X_train, y_train)
print(f"Train accuracy = {train_score*100:.2f} %")

Train accuracy = 86.65 %


In [175]:
# get precision / recall / f1
y_pred = clf.predict(X_train)
print(classification_report(y_train, y_pred, target_names=["AD", "CN"]))

              precision    recall  f1-score   support

          AD       0.91      0.84      0.87       187
          CN       0.82      0.90      0.86       150

    accuracy                           0.87       337
   macro avg       0.87      0.87      0.87       337
weighted avg       0.87      0.87      0.87       337



### Now we can evaluate on dev set

In [176]:
X_dev_tfidf = vectorizer.transform(dev_df["text_lower"])
y_dev = dev_df["target"]
print("X_dev shape =", X_dev_tfidf.shape)
print("y_dev len =", len(y_dev))

X_dev shape = (115, 1438)
y_dev len = 115


In [177]:
# create X_dev embeddings
print("Creating text embeddings...")
X_dev_embed = vectorizer_embed.encode(dev_df["text_lower"])
print("Completed")

Creating text embeddings...


Completed


In [178]:
print(type(X_dev_embed))
print(X_dev_embed.shape)

<class 'numpy.ndarray'>
(115, 768)


In [179]:
# double check Tfidf vector
print(type(X_dev_tfidf))
print(X_dev_tfidf.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(115, 1438)


In [180]:
X_dev = combine_vectors(X_dev_tfidf, X_dev_embed)
print(X_dev.shape)

(115, 2206)


In [181]:
dev_score = clf.score(X_dev, y_dev)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 79.13 %


In [182]:
y_pred = clf.predict(X_dev)
print(classification_report(y_dev, y_pred, target_names=["AD", "CN"]))

              precision    recall  f1-score   support

          AD       0.83      0.78      0.81        64
          CN       0.75      0.80      0.77        51

    accuracy                           0.79       115
   macro avg       0.79      0.79      0.79       115
weighted avg       0.79      0.79      0.79       115



In [183]:
# extract AD tuning score
print("Tuning score =", tuning_score)
classification_report(y_dev, y_pred, output_dict=True)["AD"][tuning_score]

Tuning score = f1-score


0.8064516129032258

### Hyperparameter tuning
We can adjust a few parameters on the Tfidf vectorizer and class_weight in the Logistic regression

In [184]:
# following are sets of parameters for tuning
tfidf_params_max_features = [None, 1000, 5000, 10000]
tfidf_ngrams = [(1, 1), (1, 2), (1, 3), (1, 4), (2, 3), (2, 4)]

lr_class_weights = [None,
                    "balanced", 
                    {"AD":0.65, "CN":0.35},
                    {"AD":0.4, "CN":0.6}, 
                    {"AD":0.45, "CN":0.55}, 
                    {"AD":0.5, "CN":0.5}, 
                    {"AD":0.55, "CN":0.45},
                    {"AD":0.6, "CN":0.4},
                    {"AD":0.65, "CN":0.35}
                   ]

best_params = dict()

best_dev_score = 0      # overall accuracy score
best_avg_dev_score = 0  # average between accuracy and tuning score (recall / f1 / precision) 

print(f"Hyper parameter tuning on accuracy and AD {tuning_score}...")
for max_feat in tfidf_params_max_features:
    for ngram in tfidf_ngrams:
        for class_weights in lr_class_weights:
            print(f"Params, max_feat = {max_feat}, ngram = {ngram}, class_weights = {class_weights}")
        
            vectorizer = TfidfVectorizer(max_features=max_feat, ngram_range=ngram)

            X_train_tfidf = vectorizer.fit_transform(train_df["text_lower"])
            X_train = combine_vectors(X_train_tfidf, X_train_embed)
        
            # create logistic regression classifier
            clf = LogisticRegression(random_state=42,
                                     class_weight= class_weights 
                                    ).fit(X_train, y_train)
            
            # getting score on train set
            train_score = clf.score(X_train, y_train)
            print(f"Train accuracy = {train_score*100:.2f} %")            

            y_pred_train = clf.predict(X_train)
            train_tune_score = classification_report(y_train, y_pred_train, output_dict=True)["AD"][tuning_score]
            print(f"Train {tuning_score} = {train_tune_score*100:.2f} %")
    
            # getting score on dev set
            X_dev_tfidf = vectorizer.transform(dev_df["text_lower"])
            X_dev = combine_vectors(X_dev_tfidf, X_dev_embed)
            y_dev = dev_df["target"]
            
            dev_score = clf.score(X_dev, y_dev)
            print(f"Dev accuracy = {dev_score*100:.2f} %")

            # extract AD tuning rate (recall / f1)
            y_pred_dev = clf.predict(X_dev)
            dev_tuning_score = classification_report(y_dev, y_pred_dev, output_dict=True)["AD"][tuning_score]
            print(f"Dev {tuning_score} = {dev_tuning_score*100:.2f} %")

            dev_avg_score = (dev_score + dev_tuning_score) / 2

            if dev_avg_score > best_avg_dev_score:
                print("**** New best average dev score found =", dev_avg_score)
                
                best_dev_score = dev_score
                best_avg_dev_score = dev_avg_score
                best_params["max_feat"] = max_feat
                best_params["class_weights"] = class_weights
                best_params["ngram"] = ngram
                
            print("-------")
                    

print("Completed")
print("Best dev score =", best_dev_score)
print("Best average dev score =", best_avg_dev_score)

print("Best params =", best_params)

Hyper parameter tuning on accuracy and AD f1-score...
Params, max_feat = None, ngram = (1, 1), class_weights = None
Train accuracy = 88.43 %
Train f1-score = 89.66 %
Dev accuracy = 77.39 %
Dev f1-score = 79.69 %
**** New best average dev score found = 0.7853940217391304
-------
Params, max_feat = None, ngram = (1, 1), class_weights = balanced
Train accuracy = 86.65 %
Train f1-score = 87.47 %
Dev accuracy = 79.13 %
Dev f1-score = 80.65 %
**** New best average dev score found = 0.7988779803646564
-------
Params, max_feat = None, ngram = (1, 1), class_weights = {'AD': 0.65, 'CN': 0.35}
Train accuracy = 70.33 %
Train f1-score = 78.90 %
Dev accuracy = 67.83 %
Dev f1-score = 76.73 %
-------
Params, max_feat = None, ngram = (1, 1), class_weights = {'AD': 0.4, 'CN': 0.6}
Train accuracy = 83.38 %
Train f1-score = 83.53 %
Dev accuracy = 77.39 %
Dev f1-score = 77.19 %
-------
Params, max_feat = None, ngram = (1, 1), class_weights = {'AD': 0.45, 'CN': 0.55}
Train accuracy = 83.68 %
Train f1-score 

In [185]:
# if needed, uncomment the lines below for the fine-tuned best params
# manually set best_params for consistency
best_params["max_feat"] = 10000
best_params["class_weights"] = "balanced"
best_params["ngram"] = (1, 2)

### Create tfidf vectorizer and logistic regression classifier with new best params

In [186]:
vectorizer = TfidfVectorizer(max_features=best_params["max_feat"], ngram_range=best_params["ngram"])

X_train_tfidf = vectorizer.fit_transform(train_df["text_lower"])
X_train = combine_vectors(X_train_tfidf, X_train_embed)

clf = LogisticRegression(random_state =42,
                         class_weight = best_params["class_weights"]
                        ).fit(X_train, y_train)

# double check dev score
X_dev_tfidf = vectorizer.transform(dev_df["text_lower"])
y_dev = dev_df["target"]

X_dev = combine_vectors(X_dev_tfidf, X_dev_embed)


dev_score = clf.score(X_dev, y_dev)
print(f"Train accuracy = {clf.score(X_train, y_train)*100:.2f} %")
print(f"Dev accuracy = {dev_score*100:.2f} %")

Train accuracy = 91.10 %
Dev accuracy = 79.13 %


In [187]:
print(classification_report(y_train, clf.predict(X_train)))

              precision    recall  f1-score   support

          AD       0.94      0.90      0.92       187
          CN       0.88      0.93      0.90       150

    accuracy                           0.91       337
   macro avg       0.91      0.91      0.91       337
weighted avg       0.91      0.91      0.91       337



In [188]:
print(classification_report(y_dev, clf.predict(X_dev)))

              precision    recall  f1-score   support

          AD       0.84      0.77      0.80        64
          CN       0.74      0.82      0.78        51

    accuracy                           0.79       115
   macro avg       0.79      0.79      0.79       115
weighted avg       0.80      0.79      0.79       115



### Process prediction for test set

In [189]:
test_fname = data_path + "pitt/pitt_test_txt.csv"

test_df = pd.read_csv(test_fname)
print("Test df =", len(test_df))
test_df["text_lower"] = test_df["text"].apply(lambda text: text.lower().strip())
test_df.head()

Test df = 97


,file_id,target,text,mmse,id,split_name,text_lower
0,001-0,AD,mhm .\nalright .\nthere's &-um a young boy tha...,18.0,1,test,mhm .\nalright .\nthere's &-um a young boy tha...
1,001-2,AD,mhm .\nthere's a young boy &-uh going in a coo...,11.0,1,test,mhm .\nthere's a young boy &-uh going in a coo...
2,023-0,AD,(..) well I see the mother .\nwhat's she doin(...,16.0,23,test,(..) well i see the mother .\nwhat's she doin(...
3,023-2,AD,well you see <the little> the little boy stea...,3.0,23,test,well you see <the little> the little boy stea...
4,039-0,AD,okay we see &-um first of all &-uh the sink is...,28.0,39,test,okay we see &-um first of all &-uh the sink is...


In [190]:
# add summarization
print("Performing summarization")
print("Test set...")

test_df["text_lower"] = test_df["text"].apply(summ_f)

print("Completed")
test_df.head()

Performing summarization
Test set...
Completed


,file_id,target,text,mmse,id,split_name,text_lower
0,001-0,AD,mhm .\nalright .\nthere's &-um a young boy tha...,18.0,1,test,mhm .\nalright .\nthere's &-um a young boy tha...
1,001-2,AD,mhm .\nthere's a young boy &-uh going in a coo...,11.0,1,test,mhm .\nthere's a young boy &-uh going in a coo...
2,023-0,AD,(..) well I see the mother .\nwhat's she doin(...,16.0,23,test,(..) well i see the mother .\nwhat's she doin(...
3,023-2,AD,well you see <the little> the little boy stea...,3.0,23,test,well you see <the little> the little boy stea...
4,039-0,AD,okay we see &-um first of all &-uh the sink is...,28.0,39,test,okay we see &-um first of all &-uh the sink is...


In [191]:
X_test_tfidf = vectorizer.transform(test_df["text_lower"])


In [192]:
# create X_test embeddings
print("Creating text embeddings...")
X_test_embed = vectorizer_embed.encode(test_df["text_lower"])
print("Completed")

Creating text embeddings...
Completed


In [193]:
X_test = combine_vectors(X_test_tfidf, X_test_embed)

print(X_test.shape)

(97, 10768)


In [194]:
test_df["prediction"] = clf.predict(X_test)
test_df.head()

,file_id,target,text,mmse,id,split_name,text_lower,prediction
0,001-0,AD,mhm .\nalright .\nthere's &-um a young boy tha...,18.0,1,test,mhm .\nalright .\nthere's &-um a young boy tha...,AD
1,001-2,AD,mhm .\nthere's a young boy &-uh going in a coo...,11.0,1,test,mhm .\nthere's a young boy &-uh going in a coo...,AD
2,023-0,AD,(..) well I see the mother .\nwhat's she doin(...,16.0,23,test,(..) well i see the mother .\nwhat's she doin(...,AD
3,023-2,AD,well you see <the little> the little boy stea...,3.0,23,test,well you see <the little> the little boy stea...,AD
4,039-0,AD,okay we see &-um first of all &-uh the sink is...,28.0,39,test,okay we see &-um first of all &-uh the sink is...,AD


In [195]:
correct_count = len(test_df.query("target == prediction"))
print(f"Correct count = {correct_count}, accuracy = {correct_count*100/len(test_df):.0f} %")

Correct count = 89, accuracy = 92 %


In [196]:
# check if the test set has 'target' column
if "target" in test_df.columns:
    print(classification_report(test_df["target"], test_df["prediction"]))

              precision    recall  f1-score   support

          AD       0.93      0.93      0.93        55
          CN       0.90      0.90      0.90        42

    accuracy                           0.92        97
   macro avg       0.92      0.92      0.92        97
weighted avg       0.92      0.92      0.92        97



In [197]:
# write output to pitt_test_predictions.csv
fname_out = data_path + "pitt/pitt_test_predictions.csv"
print("Writing to file =", fname_out)
test_df.to_csv(fname_out,
               index=False
              )
print("Completed")

Writing to file = ../../../data/pitt/pitt_test_predictions.csv
Completed


### Train Ridge 

We can now train a ridge regression model

In [200]:
print("Training ridge pipeline...")
train_df = train_df[train_df["mmse"].notna()]
pipeline = make_pipeline(vectorizer, RidgeCV(cv=10))
pipeline.fit(train_df["text_lower"], train_df["mmse"])
print("Completed")
y_pred = pipeline.predict(train_df["text_lower"])
print("Training RMSE: ", np.sqrt(mean_squared_error(train_df["mmse"], y_pred)))

Training ridge pipeline...
Completed
Training RMSE:  0.5068770455013972


In [201]:
dev_df = dev_df[dev_df["mmse"].notna()]
y_pred = pipeline.predict(dev_df["text_lower"])
print("Dev RMSE: ", np.sqrt(mean_squared_error(dev_df["mmse"], y_pred)))

Dev RMSE:  4.5563493990897355


In [202]:
test_df = test_df[test_df["mmse"].notna()]
y_test = pipeline.predict(test_df["text_lower"])
print("Test RMSE: ", np.sqrt(mean_squared_error(test_df["mmse"], y_test)))

Test RMSE:  5.469510831437423
